<a href="https://colab.research.google.com/github/Suhana-09-2005/NLP/blob/main/nlplab14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# First, let's download the SMS Spam Collection dataset.
!wget -nc https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip
!unzip -n smsspamcollection.zip

import pandas as pd

# The dataset is typically in a TSV format (tab-separated values) without a header.
df = pd.read_csv('SMSSpamCollection', sep='\t', header=None, names=['label', 'text'])

print("Dataset loaded successfully. Here are the first 5 rows:")
display(df.head())

--2026-03-27 17:09:37--  https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘smsspamcollection.zip’

smsspamcollection.z     [ <=>                ] 198.65K  1.01MB/s    in 0.2s    

2026-03-27 17:09:38 (1.01 MB/s) - ‘smsspamcollection.zip’ saved [203415]

Archive:  smsspamcollection.zip
  inflating: SMSSpamCollection       
  inflating: readme                  
Dataset loaded successfully. Here are the first 5 rows:


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


This dataset contains two columns: `label` (indicating 'ham' for legitimate messages and 'spam' for spam messages) and `text` (the content of the SMS message). Let's check the distribution of these labels.

In [2]:
print("\nLabel distribution:")
display(df['label'].value_counts())


Label distribution:


,count
label,
ham,4825
spam,747


### Text Preprocessing

Now, let's preprocess the text data. This typically involves several steps to clean and standardize the text:

1.  **Lowercasing**: Convert all text to lowercase to treat words like 'Free' and 'free' as the same.
2.  **Removing Punctuation**: Get rid of special characters and punctuation that don't add semantic value.
3.  **Tokenization**: Break down the text into individual words (tokens).
4.  **Removing Stop Words**: Eliminate common words (like 'the', 'is', 'a') that often don't carry much meaning for classification.
5.  **Stemming/Lemmatization** (Optional, but often useful): Reduce words to their root form.

For this task, we will focus on lowercasing, removing punctuation, and tokenization first. We'll use `nltk` for tokenization and potentially stop word removal.

In [6]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download NLTK data if not already present
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    # Explicitly download 'punkt_tab' as indicated by the error message
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))
punctuations = r'[!"#$%&()*+,-./:;<=>?@[\]^_`{|}~]'
stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower() # Lowercasing
    text = re.sub(punctuations, '', text) # Remove punctuation
    tokens = nltk.word_tokenize(text) # Tokenization
    # Optionally remove stop words and stem. For now, let's just keep tokens.
    # tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

df['processed_text'] = df['text'].apply(preprocess_text)

print("Original text vs. Processed text (first 5 rows):")
comparison_df = df[['text', 'processed_text']].head()
display(comparison_df)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Original text vs. Processed text (first 5 rows):


,text,processed_text
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,"Nah I don't think he goes to usf, he lives aro...",nah i do n't think he goes to usf he lives aro...


In [7]:
from collections import Counter
import torch

# 1. Create Vocabulary
all_words = ' '.join(df['processed_text']).split()
word_counts = Counter(all_words)
vocabulary = {word: i + 2 for i, (word, _) in enumerate(word_counts.most_common())}
vocabulary['<PAD>'] = 0 # Padding token
vocabulary['<UNK>'] = 1 # Unknown word token

# Reverse mapping for later use (optional, but good for debugging)
idx_to_word = {i: word for word, i in vocabulary.items()}

print(f"Vocabulary size: {len(vocabulary)}")
print("Top 10 words in vocabulary:", list(word_counts.most_common(10)))

# 2. Encode Text
def encode_text(text, vocab, max_len):
    tokens = text.split()
    encoded_tokens = [vocab.get(word, vocab['<UNK>']) for word in tokens]

    # Pad or truncate sequences
    if len(encoded_tokens) < max_len:
        padded_tokens = encoded_tokens + [vocab['<PAD>']] * (max_len - len(encoded_tokens))
    else:
        padded_tokens = encoded_tokens[:max_len]
    return padded_tokens

# Determine max_len (e.g., 90th percentile of message lengths)
message_lengths = df['processed_text'].apply(lambda x: len(x.split()))
MAX_LEN = int(message_lengths.quantile(0.95)) # Using 95th percentile as a robust choice

print(f"Chosen MAX_LEN for padding: {MAX_LEN}")

df['encoded_text'] = df['processed_text'].apply(lambda x: encode_text(x, vocabulary, MAX_LEN))

print("\nOriginal text, processed text, and encoded text for the first 3 samples:")
for i in range(3):
    print(f"Original: {df['text'][i]}")
    print(f"Processed: {df['processed_text'][i]}")
    print(f"Encoded: {df['encoded_text'][i]}")
    print("-" * 30)

# Map labels to numerical values
label_mapping = {'ham': 0, 'spam': 1}
df['label_encoded'] = df['label'].map(label_mapping)

print("\nEncoded labels (first 5):")
display(df[['label', 'label_encoded']].head())

Vocabulary size: 9604
Top 10 words in vocabulary: [('i', 2875), ('to', 2251), ('you', 2216), ('a', 1442), ('the', 1333), ('u', 1151), ('and', 971), ('is', 904), ('in', 888), ('me', 802)]
Chosen MAX_LEN for padding: 33

Original text, processed text, and encoded text for the first 3 samples:
Original: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Processed: go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat
Encoded: [49, 441, 4386, 799, 722, 686, 68, 10, 1251, 95, 123, 353, 1252, 158, 2898, 1253, 64, 57, 4387, 134, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
------------------------------
Original: Ok lar... Joking wif u oni...
Processed: ok lar joking wif u oni
Encoded: [50, 316, 1404, 442, 7, 1835, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
------------------------------
Original: Free entry in 2 a wkly comp to win FA Cup final tkts 21s

,label,label_encoded
0,ham,0
1,ham,0
2,spam,1
3,ham,0
4,ham,0


### Vocabulary Creation & Encoding

To feed our text data into a neural network, we need to convert words into numerical representations. This process involves building a vocabulary of all unique words in our dataset and then mapping each word to a unique integer ID. Finally, each text message will be transformed into a sequence of these integer IDs. Since neural networks require fixed-size inputs, we'll also pad these sequences to a uniform length.

We will determine the `MAX_LEN` for padding based on the longest message in the dataset or a reasonable percentile to avoid excessively long sequences.

In [11]:
from collections import Counter
import torch
import numpy as np

# 1. Create Vocabulary
all_words = ' '.join(df['processed_text'].astype(str)).split()
word_counts = Counter(all_words)

# Reserve 0 for PAD, 1 for UNK
vocabulary = {word: i + 2 for i, (word, _) in enumerate(word_counts.most_common())}
vocabulary['<PAD>'] = 0
vocabulary['<UNK>'] = 1

# Reverse mapping (optional)
idx_to_word = {i: word for word, i in vocabulary.items()}

print(f"Vocabulary size: {len(vocabulary)}")
print("Top 10 words:", word_counts.most_common(10))


# 2. Encode Text Function
def encode_text(text, vocab, max_len):
    tokens = str(text).split()   # ensure string
    encoded = [vocab.get(word, vocab['<UNK>']) for word in tokens]

    # Padding / Truncation
    if len(encoded) < max_len:
        encoded += [vocab['<PAD>']] * (max_len - len(encoded))
    else:
        encoded = encoded[:max_len]

    return encoded


# 3. Determine MAX_LEN (95th percentile)
message_lengths = df['processed_text'].astype(str).apply(lambda x: len(x.split()))
MAX_LEN = int(np.percentile(message_lengths, 95))

print(f"Chosen MAX_LEN: {MAX_LEN}")


# 4. Apply Encoding
df['encoded_text'] = df['processed_text'].apply(lambda x: encode_text(x, vocabulary, MAX_LEN))


# 5. Convert to PyTorch tensors (IMPORTANT for CNN)
X = torch.tensor(df['encoded_text'].tolist(), dtype=torch.long)


# 6. Encode Labels
label_mapping = {'ham': 0, 'spam': 1}
df['label_encoded'] = df['label'].map(label_mapping)

y = torch.tensor(df['label_encoded'].values, dtype=torch.long)


# 7. Debug Output
print("\nSample Data:")
for i in range(3):
    print(f"Original: {df['text'].iloc[i]}")
    print(f"Processed: {df['processed_text'].iloc[i]}")
    print(f"Encoded: {df['encoded_text'].iloc[i]}")
    print("-" * 30)

print("\nEncoded labels:")
print(df[['label', 'label_encoded']].head())

Vocabulary size: 9604
Top 10 words: [('i', 2875), ('to', 2251), ('you', 2216), ('a', 1442), ('the', 1333), ('u', 1151), ('and', 971), ('is', 904), ('in', 888), ('me', 802)]
Chosen MAX_LEN: 33

Sample Data:
Original: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Processed: go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat
Encoded: [49, 441, 4386, 799, 722, 686, 68, 10, 1251, 95, 123, 353, 1252, 158, 2898, 1253, 64, 57, 4387, 134, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
------------------------------
Original: Ok lar... Joking wif u oni...
Processed: ok lar joking wif u oni
Encoded: [50, 316, 1404, 442, 7, 1835, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
------------------------------
Original: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 084528

### Build 1D CNN Model

Now we will define our 1D Convolutional Neural Network (CNN) model. This model will typically consist of:

1.  **Embedding Layer**: To convert word indices into dense vector representations.
2.  **Convolutional Layer (Conv1D)**: To extract local features from the text sequences.
3.  **Pooling Layer**: To reduce the dimensionality and capture the most important features.
4.  **Fully Connected Layer**: For classification based on the extracted features.

In [10]:
import torch.nn as nn
import torch.nn.functional as F

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_filters, filter_sizes, output_dim, dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0) # <PAD> token is 0

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim,
                      out_channels=num_filters,
                      kernel_size=fs)
            for fs in filter_sizes
        ])

        self.fc = nn.Linear(len(filter_sizes) * num_filters, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        # text = [batch size, sent len]

        embedded = self.embedding(text)

        # embedded = [batch size, sent len, emb dim]

        # PyTorch's Conv1D expects input of shape (batch_size, in_channels, sequence_length)
        # We need to permute the embedded tensor
        embedded = embedded.permute(0, 2, 1)

        # embedded = [batch size, emb dim, sent len]

        conved = [F.relu(conv(embedded)) for conv in self.convs]

        # conved_n = [batch size, num_filters, output_len]

        pooled = [F.max_pool1d(conv, conv.shape[2]).squeeze(2) for conv in conved]

        # pooled_n = [batch size, num_filters]

        cat = self.dropout(torch.cat(pooled, dim=1))

        # cat = [batch size, num_filters * len(filter_sizes)]

        return self.fc(cat)

# Model Hyperparameters
VOCAB_SIZE = len(vocabulary) # From previous step
EMBEDDING_DIM = 100
NUM_FILTERS = 100
FILTER_SIZES = [3, 4, 5] # Example filter sizes
OUTPUT_DIM = 1 # Binary classification (spam or ham)
DROPOUT = 0.5

# Instantiate the model
model = TextCNN(VOCAB_SIZE, EMBEDDING_DIM, NUM_FILTERS, FILTER_SIZES, OUTPUT_DIM, DROPOUT)

print(model)

# Count trainable parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')


TextCNN(
  (embedding): Embedding(9604, 100, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(100, 100, kernel_size=(3,), stride=(1,))
    (1): Conv1d(100, 100, kernel_size=(4,), stride=(1,))
    (2): Conv1d(100, 100, kernel_size=(5,), stride=(1,))
  )
  (fc): Linear(in_features=300, out_features=1, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)
The model has 1,081,001 trainable parameters


### Create PyTorch Dataset Class

To effectively train our deep learning model using PyTorch, we need a custom `Dataset` class. This class will encapsulate our encoded text and labels, providing a standardized way to access individual samples. It will also be used with `DataLoader` to create batches of data for training.

In [9]:
from torch.utils.data import Dataset, DataLoader

class SMSDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = torch.tensor(self.texts[idx], dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return text, label

# Create instances of the custom dataset
train_dataset = SMSDataset(X_train, y_train)
test_dataset = SMSDataset(X_test, y_test)

print(f"Training dataset created with {len(train_dataset)} samples.")
print(f"Test dataset created with {len(test_dataset)} samples.")

# Example of accessing an item
first_train_text, first_train_label = train_dataset[0]
print(f"\nFirst training sample - Text (encoded): {first_train_text}")
print(f"First training sample - Label: {first_train_label}")

Training dataset created with 4457 samples.
Test dataset created with 1115 samples.

First training sample - Text (encoded): tensor([ 66,  37,   4, 333, 633,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0])
First training sample - Label: 0


### Train-Test Split

Before training any machine learning model, it's essential to split the data into training and testing sets. The training set will be used to teach the model, and the test set will be used to evaluate how well the model generalizes to new, unseen data.

In [8]:
from sklearn.model_selection import train_test_split

X = list(df['encoded_text'].values)
y = list(df['label_encoded'].values)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {len(X_train)} samples")
print(f"Test set size: {len(X_test)} samples")
print(f"First 5 encoded text samples (training): {X_train[:5]}")
print(f"First 5 corresponding labels (training): {y_train[:5]}")

Training set size: 4457 samples
Test set size: 1115 samples
First 5 encoded text samples (training): [[66, 37, 4, 333, 633, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [30, 2, 108, 63, 44, 25, 6424, 6425, 39, 6426, 6427, 6428, 6429, 6430, 6431, 6432, 42, 6433, 6434, 6435, 46, 2078, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [50, 9465, 2, 683, 24, 9466, 296, 2, 864, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 62, 86, 28, 2870, 156, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 62, 86, 39, 2, 30, 875, 77, 10, 5, 274, 57, 115, 242, 3, 109, 196, 16, 116, 8722, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
First 5 corresponding labels (training): [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
